# Import One Topic — Pipeline Notebook

Run each cell in order. Edit the **Configuration** cell below to set your topic, database, and parameters.

Each step can be re-run independently. Progress is checkpointed automatically.

In [ ]:
# ==================== CONFIGURATION ====================
# Edit these variables before running the pipeline.

TOPIC = "Particle Physics"            # Topic name, e.g. "Astrophysics", "Nuclear Physics"
DB    = "papers_particle_physics.db"  # SQLite DB filename
EMAIL = "tom.hirsch3000@gmail.com"    # Email for OpenAlex polite pool
API_KEY = None                        # OpenAlex API key (None = use polite pool only)

# --- Import size control ---
TOP_N_PAPERS = 500    # Import only the top N most-cited papers (0 = no limit, import everything)
                      # When set, ignores year batching and does a single query sorted by citations

# --- Import batching (only used when TOP_N_PAPERS = 0) ---
YEAR_START       = 1800   # First publication year to import
YEAR_END         = 2026   # Last publication year to import
YEAR_BATCH_SIZE  = 1      # Years per import batch (use 10+ for sparse topics)
PAPERS_PER_BATCH = 0      # Max papers per year-range batch (0 = no limit)

# --- AI sampling ---
AI_SAMPLE     = 500   # Total papers to run AI on (0 = all)
AI_BATCH_SIZE = 200   # Papers per AI subprocess call (reduce if Ollama is slow)

# --- Visualization ---
MIN_CITATIONS = 0     # Min citations to include in visualization

# --- Step control ---
SKIP_STEPS  = []      # Steps to skip entirely, e.g. [3, 7]
FORCE_STEPS = []      # Steps to force re-run, e.g. [4]
ONLY_STEP   = None    # Run only this step number (None = run all)
RESET_IMPORT    = False
RESET_CITATIONS = False
SKIP_MISLABEL   = True

print(f"Configuration loaded: topic={TOPIC!r}, db={DB!r}")
if TOP_N_PAPERS > 0:
    print(f"Import mode: top {TOP_N_PAPERS} papers by citation count")
else:
    print(f"Import mode: all papers, year batches {YEAR_START}-{YEAR_END}")

In [ ]:
import datetime
import glob
import json
import sqlite3
import subprocess
import sys
import os
from pathlib import Path

# In a notebook, use the notebook's directory instead of __file__
SCRIPT_DIR = Path(os.getcwd())
DATA_DIR = SCRIPT_DIR / "data"
CHECKPOINT_VERSION = 2

# Derived values
slug = TOPIC.lower().replace(" ", "_")
DATA_DIR.mkdir(exist_ok=True)
nodes_out = f"data/{slug}_nodes.json"
edges_out = f"data/{slug}_edges.json"
meta_out  = f"data/{slug}_metadata.json"
FRONTEND_DIR = str(SCRIPT_DIR.parent / "arxiv-3d-frontend" / "public")

print(f"SCRIPT_DIR : {SCRIPT_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"Output     : {nodes_out}, {edges_out}, {meta_out}")
print(f"Frontend   : {FRONTEND_DIR}")

In [ ]:
# ---------------------------------------------------------------------------
# Helper functions (checkpoint, DB queries, subprocess runner)
# ---------------------------------------------------------------------------

def checkpoint_path(db):
    return SCRIPT_DIR / f"{Path(db).stem}_checkpoint.json"

def load_checkpoint(db):
    p = checkpoint_path(db)
    if p.exists():
        try:
            with open(p) as f:
                data = json.load(f)
            if data.get("version") == CHECKPOINT_VERSION:
                return data
            print(f"[checkpoint] Version mismatch — starting fresh")
        except Exception as e:
            print(f"[checkpoint] Could not read {p}: {e} — starting fresh")
    return {"version": CHECKPOINT_VERSION, "import_batches_done": [], "steps_done": [], "ai_processed": 0}

def save_checkpoint(db, cp):
    cp["updated"] = datetime.datetime.now().isoformat(timespec="seconds")
    with open(checkpoint_path(db), "w") as f:
        json.dump(cp, f, indent=2)

def mark_step_done(db, cp, step):
    if step not in cp["steps_done"]:
        cp["steps_done"].append(step)
    save_checkpoint(db, cp)

def mark_import_batch_done(db, cp, batch_key_str):
    if batch_key_str not in cp["import_batches_done"]:
        cp["import_batches_done"].append(batch_key_str)
    save_checkpoint(db, cp)

def count_unprocessed_ai(db):
    db_path = SCRIPT_DIR / db
    if not db_path.exists():
        return 0
    try:
        conn = sqlite3.connect(str(db_path))
        cols = {r[1] for r in conn.execute("PRAGMA table_info(papers)").fetchall()}
        if "AI_field_list" not in cols:
            n = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        else:
            n = conn.execute("""
                SELECT COUNT(*) FROM papers
                WHERE AI_field_list IS NULL OR AI_field_list = '[]'
                   OR AI_summary IS NULL OR TRIM(AI_summary) = ''
            """).fetchone()[0]
        conn.close()
        return n
    except Exception as e:
        print(f"[warn] Could not query DB for AI count: {e}")
        return 0

def count_total_papers(db):
    db_path = SCRIPT_DIR / db
    if not db_path.exists():
        return 0
    try:
        conn = sqlite3.connect(str(db_path))
        n = conn.execute("SELECT COUNT(*) FROM papers").fetchone()[0]
        conn.close()
        return n
    except Exception:
        return 0

def run_cmd(cmd, label, fatal=True):
    print(f"\n{'='*60}")
    print(f"  {label}")
    print(f"  $ {' '.join(str(c) for c in cmd)}")
    print(f"{'='*60}")
    result = subprocess.run([str(c) for c in cmd], cwd=SCRIPT_DIR)
    if result.returncode != 0:
        print(f"\n[FAIL] Exit code {result.returncode}")
        if fatal:
            raise RuntimeError(f"Step failed: {label}. Fix the issue and re-run this cell.")
        return False
    print(f"[OK] Done.")
    return True

def make_year_batches(year_start, year_end, batch_size):
    batches = []
    y = year_start
    while y <= year_end:
        end = min(y + batch_size - 1, year_end)
        batches.append((y, end))
        y = end + 1
    return batches

def batch_key(from_year, to_year):
    return f"{from_year}-{to_year}"

def find_galaxy_args():
    node_files = sorted(glob.glob(str(DATA_DIR / "*_nodes.json")))
    galaxies = []
    for idx, nodes_path in enumerate(node_files, start=1):
        nodes_file = Path(nodes_path).name
        base = nodes_file.replace("_nodes.json", "")
        edges_file = f"{base}_edges.json"
        meta_file = f"{base}_metadata.json"
        name = base.replace("_", " ").title()
        if not (DATA_DIR / edges_file).exists() or not (DATA_DIR / meta_file).exists():
            continue
        galaxies.append(f"{idx}:{name}:data/{nodes_file}:data/{edges_file}:data/{meta_file}")
    return galaxies

def should_run(step, cp):
    if ONLY_STEP is not None:
        return step == ONLY_STEP
    if step in SKIP_STEPS:
        return False
    if step in FORCE_STEPS:
        return True
    if step in cp["steps_done"]:
        print(f"[checkpoint] Step {step} already done — skipping. Add {step} to FORCE_STEPS to re-run.")
        return False
    return True

# Load checkpoint
cp = load_checkpoint(DB)
cp.setdefault("topic", TOPIC)
cp.setdefault("db", DB)
save_checkpoint(DB, cp)

print(f"Checkpoint : {checkpoint_path(DB).name}")
print(f"Steps done : {cp['steps_done']}")
print(f"Import batches done: {len(cp['import_batches_done'])}")
print(f"AI papers processed: {cp['ai_processed']}")
print(f"Total papers in DB : {count_total_papers(DB)}")

## Step 1 — Import papers from OpenAlex
- **TOP_N_PAPERS > 0**: imports the top N most-cited papers in a single query (no year batching)
- **TOP_N_PAPERS = 0**: downloads all papers in year-range batches, checkpointed per batch

In [ ]:
# STEP 1 — Import papers
if should_run(1, cp):
    if TOP_N_PAPERS > 0:
        # ---------- Top-N mode: single query, sorted by cited_by_count:desc ----------
        print(f"[step 1] Importing top {TOP_N_PAPERS} most-cited papers for '{TOPIC}'...")
        cmd = [
            sys.executable, "import_openalex.py",
            "--topic-name", TOPIC,
            "--db", DB,
            "--email", EMAIL,
            "--sample", str(TOP_N_PAPERS),
        ]
        if API_KEY:
            cmd += ["--api-key", API_KEY]
        if RESET_IMPORT:
            cmd.append("--reset")
        run_cmd(cmd, f"1 — Import top {TOP_N_PAPERS} by citations")
        mark_step_done(DB, cp, 1)
        print(f"\n[step 1] Import complete. Total papers: {count_total_papers(DB)}")
    else:
        # ---------- Year-batch mode ----------
        batches = make_year_batches(YEAR_START, YEAR_END, YEAR_BATCH_SIZE)
        done_keys = set(cp["import_batches_done"])

        if 1 in FORCE_STEPS:
            print("[info] FORCE_STEPS includes 1: clearing import batch history")
            cp["import_batches_done"] = []
            done_keys = set()
            save_checkpoint(DB, cp)

        pending = [(f, t) for (f, t) in batches if batch_key(f, t) not in done_keys]
        print(f"[step 1] {len(batches)} year-range batches total, "
              f"{len(done_keys)} already done, {len(pending)} to go.")

        reset_import = RESET_IMPORT
        for from_y, to_y in pending:
            bk = batch_key(from_y, to_y)
            cmd = [
                sys.executable, "import_openalex.py",
                "--topic-name", TOPIC,
                "--db", DB,
                "--email", EMAIL,
                "--from-year", str(from_y),
                "--to-year",   str(to_y),
            ]
            if API_KEY:
                cmd += ["--api-key", API_KEY]
            if PAPERS_PER_BATCH > 0:
                cmd += ["--sample", str(PAPERS_PER_BATCH)]
            if reset_import:
                cmd.append("--reset")
                reset_import = False

            ok = run_cmd(cmd, f"1 — Import {from_y}–{to_y}", fatal=False)
            if ok:
                mark_import_batch_done(DB, cp, bk)
                print(f"[checkpoint] Batch {bk} done. Total papers: {count_total_papers(DB)}")
            else:
                print(f"[warn] Batch {bk} failed. Re-run this cell to resume.")
                break

        mark_step_done(DB, cp, 1)
        print(f"\n[step 1] Import complete. Total papers: {count_total_papers(DB)}")
else:
    print(f"Step 1 skipped. Papers in DB: {count_total_papers(DB)}")

## Step 2 — Build citation edges

In [ ]:
# STEP 2 — Build citation edges
if should_run(2, cp):
    cmd = [sys.executable, "rebuild_citations_openalex.py", "--db", DB]
    if RESET_CITATIONS:
        cmd.append("--reset")
    run_cmd(cmd, "2 — Rebuild citation edges")
    mark_step_done(DB, cp, 2)
else:
    print("Step 2 skipped.")

## Step 3 — Fetch missing abstracts (Semantic Scholar / arXiv)

In [ ]:
# STEP 3 — Fetch missing abstracts
if should_run(3, cp):
    run_cmd(
        [sys.executable, "fetch_abstracts_s2_arxiv.py", "--db", DB],
        "3 — Fetch missing abstracts (Semantic Scholar / arXiv)"
    )
    mark_step_done(DB, cp, 3)
else:
    print("Step 3 skipped.")

## Step 4 — AI metadata (sampled, batched)
Runs AI classification on papers. Batched with checkpoint so you can stop/resume.

In [ ]:
# STEP 4 — AI metadata (sampled, batched loop)
if should_run(4, cp):
    target = AI_SAMPLE
    batch = AI_BATCH_SIZE

    if 4 in FORCE_STEPS:
        cp["ai_processed"] = 0
        save_checkpoint(DB, cp)

    already_done = cp["ai_processed"]
    remaining_target = (target - already_done) if target > 0 else float("inf")
    unprocessed = count_unprocessed_ai(DB)

    print(f"[step 4] AI metadata:")
    print(f"         Unprocessed papers in DB : {unprocessed}")
    print(f"         Already processed (this run): {already_done}")
    print(f"         Target (AI_SAMPLE)       : {target if target > 0 else 'unlimited'}")
    print(f"         Batch size               : {batch}")

    if remaining_target <= 0:
        print(f"[checkpoint] Already reached AI sample target ({target}). "
              f"Add 4 to FORCE_STEPS or increase AI_SAMPLE.")
    elif unprocessed == 0:
        print("[info] No unprocessed papers found — skipping AI step.")
        mark_step_done(DB, cp, 4)
    else:
        processed_this_run = 0
        while True:
            to_process = batch if target == 0 else min(batch, int(remaining_target) - processed_this_run)
            if to_process <= 0:
                break
            unprocessed = count_unprocessed_ai(DB)
            if unprocessed == 0:
                print("[info] All papers now have AI metadata.")
                break

            print(f"\n[step 4] Running AI on next {to_process} papers "
                  f"({processed_this_run + already_done} done so far, "
                  f"{unprocessed} remaining in DB)...")

            run_cmd(
                [sys.executable, "process_ai_metadata.py",
                 "--db", DB,
                 "--only-unprocessed", "1",
                 "--limit", str(to_process)],
                f"4 — AI metadata (batch of {to_process})"
            )

            processed_this_run += to_process
            cp["ai_processed"] = already_done + processed_this_run
            save_checkpoint(DB, cp)

            if target > 0 and processed_this_run >= int(remaining_target):
                print(f"[info] Reached AI sample target ({target} total).")
                break

        mark_step_done(DB, cp, 4)
        print(f"\n[step 4] AI done. Total processed: {cp['ai_processed']}")
else:
    print("Step 4 skipped.")

## Step 5 — Clean and standardize field classifications

In [ ]:
# STEP 5 — Clean and standardize fields
if should_run(5, cp):
    cmd = [
        sys.executable, "clean_and_categorize.py",
        "--db", DB,
        "--field", TOPIC,
    ]
    if SKIP_MISLABEL:
        cmd.append("--skip-mislabel")
    if AI_SAMPLE > 0:
        cmd += ["--limit", str(AI_SAMPLE)]
    run_cmd(cmd, "5 — Clean and standardize field classifications")
    mark_step_done(DB, cp, 5)
else:
    print("Step 5 skipped.")

## Step 6 — Build frontend JSON (nodes, edges, metadata)

In [ ]:
# STEP 6 — Build frontend JSON
if should_run(6, cp):
    cmd = [
        sys.executable, "build_frontend_json.py",
        "--db", DB,
        "--output-nodes", nodes_out,
        "--output-edges", edges_out,
        "--output-metadata", meta_out,
        "--min-citations", str(MIN_CITATIONS),
        "--compute-clusters",
    ]
    if AI_SAMPLE > 0:
        cmd += ["--top-n", str(AI_SAMPLE)]
    if FRONTEND_DIR:
        cmd += ["--frontend-dir", FRONTEND_DIR]
    run_cmd(cmd, "6 — Build nodes/edges JSON for frontend")
    mark_step_done(DB, cp, 6)
else:
    print("Step 6 skipped.")

## Step 7 — Rebuild universe view
Discovers all topic JSON files and builds the combined universe.json.

In [ ]:
# STEP 7 — Rebuild universe view
if should_run(7, cp):
    galaxies = find_galaxy_args()
    if not galaxies:
        print("[warn] No topic JSON files found — skipping universe build.")
    else:
        cmd = [
            sys.executable, "build_universe_json.py",
            "--email", EMAIL,
            "--output", "data/universe.json",
        ]
        if FRONTEND_DIR:
            cmd += ["--frontend-dir", FRONTEND_DIR]
        for g in galaxies:
            cmd += ["--galaxies", g]
        run_cmd(cmd, "7 — Build universe view")
    mark_step_done(DB, cp, 7)
else:
    print("Step 7 skipped.")

## Summary

In [ ]:
# Reload checkpoint and show final status
cp = load_checkpoint(DB)
print(f"{'='*60}")
print(f"  Pipeline status: {TOPIC}")
print(f"  Papers in DB     : {count_total_papers(DB)}")
print(f"  AI processed     : {cp['ai_processed']}")
print(f"  Steps done       : {sorted(cp['steps_done'])}")
print(f"{'='*60}")